# Neuron playground: Qwen3.5 4B

Ask a question and see Normal Qwen and Neuron Qwen (neuron-steered) answer side by side. Steers use lyra's recipe and syntax: `L15N3920+0.6` adds 0.6 median residual norms along neuron 3920 of layer 15.

**Settings:** Accelerator **GPU T4**, Internet **on**. Optionally use Add Input to attach the `neuron-agent` notebook output; the agent's `steers.json` then shows up as `agent: ...` presets. Press **Run All**, wait about 5 minutes for the model to load, and open the public `gradio.live` link printed at the bottom. It stays up while this session runs. If the link is blocked, use the controls in the cell above it, or `ask("...", preset="sarcastic")`.

In [ ]:
DTYPE = "fp16"           # T4 has no bf16; use "fp32" if replies come out as gibberish
KEEP_ALIVE_HOURS = 0     # only for Save & Run All (batch) runs; interactive sessions stay up on their own

In [ ]:
!pip install -q -U transformers accelerate fastapi "gradio==6.30.0"
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%%writefile /kaggle/working/neuron_server.py
"""Neuron-steering server for Qwen3.5 4B on a Kaggle GPU.

Runs on 127.0.0.1:8000. Steer recipe (lyra): a neuron n in layer L gets
dose * median(||resid leaving L||, tokens 1:) / ||W_down[:, n]|| added to its
down_proj input at every position, so dose=1 writes one median residual norm.
"""
import os, re, json, random, threading
import torch
from fastapi import FastAPI
from pydantic import BaseModel
from transformers import AutoTokenizer

MODEL_ID = os.environ.get("MODEL_ID", "Qwen/Qwen3.5-4B")
HERE = os.path.dirname(os.path.abspath(__file__))
PROMPTS = json.load(open(os.path.join(HERE, "prompts.json")))
LOCK = threading.Lock()
STEERS = []  # active steers, read by the hooks: [(layer, neuron, value, mode)]


def load_model(dtype):
    kw = dict(dtype=dtype, device_map="auto")
    try:
        from transformers import AutoModelForCausalLM
        return AutoModelForCausalLM.from_pretrained(MODEL_ID, **kw)
    except Exception as e:
        print("CausalLM load failed, trying ImageTextToText:", e)
        from transformers import AutoModelForImageTextToText
        return AutoModelForImageTextToText.from_pretrained(MODEL_ID, **kw)


tok = AutoTokenizer.from_pretrained(MODEL_ID)
tok.padding_side = "left"
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

_dt = os.environ.get("DTYPE", "fp16")
if _dt == "auto":  # fp32 needs both T4s (~18 GB); a single GPU gets fp16
    _dt = "fp32" if torch.cuda.device_count() >= 2 else "fp16"
DTYPE = {"fp16": torch.float16, "fp32": torch.float32, "bf16": torch.bfloat16}[_dt]
model = load_model(DTYPE).eval()

# Text decoder layers only (skip any vision tower).
DOWN, LAYER = {}, {}
for name, mod in model.named_modules():
    if "visual" in name or "vision" in name:
        continue
    m = re.search(r"layers\.(\d+)\.mlp\.down_proj$", name)
    if m:
        DOWN[int(m.group(1))] = mod
    m = re.search(r"layers\.(\d+)$", name)
    if m:
        LAYER[int(m.group(1))] = mod
N_LAYERS = len(DOWN)
N_NEURONS = DOWN[0].in_features
W_NORM = {L: DOWN[L].weight.float().norm(dim=0).cpu() for L in DOWN}  # ||W_down[:, n]||
print(f"{MODEL_ID}: {N_LAYERS} layers x {N_NEURONS} neurons, dtype {DTYPE}")


def chat(prompt, system_prompt=None):
    msgs = ([{"role": "system", "content": system_prompt}] if system_prompt else []) + [{"role": "user", "content": prompt}]
    return tok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False, enable_thinking=False)


def encode(texts):
    return tok(texts, return_tensors="pt", padding=True).to(model.device)


# ---- steering hooks ---------------------------------------------------------
def make_hook(L):
    def hook(mod, args):
        active = [s for s in STEERS if s[0] == L]
        if not active:
            return None
        x = args[0].clone()
        for _, n, v, mode in active:
            if mode == "set":
                x[..., n] = v
            else:
                x[..., n] += v
        return (x,) + tuple(args[1:])
    return hook


for L in DOWN:
    DOWN[L].register_forward_pre_hook(make_hook(L))


# ---- calibration: median residual norm leaving each layer ---------------------
@torch.no_grad()
def calibrate():
    norms = {L: [] for L in LAYER}
    def grab(L):
        def h(mod, args, out):
            hs = out[0] if isinstance(out, tuple) else out
            norms[L].append(hs[0, 1:].float().norm(dim=-1).cpu())
        return h
    hs = [LAYER[L].register_forward_hook(grab(L)) for L in LAYER]
    try:
        for p in PROMPTS["calibration"]:
            model(**encode([chat(p)]))
    finally:
        for h in hs:
            h.remove()
    return {L: torch.cat(v).median().item() for L, v in norms.items()}


MEDIAN = calibrate()
print("median resid norms:", {L: round(v, 1) for L, v in MEDIAN.items()})


def resolve(steers):
    out = []
    for s in steers or []:
        L, n, d = int(s["layer"]), int(s["neuron"]), float(s["dose"])
        out.append((L, n, d * MEDIAN[L] / W_NORM[L][n].item(), s.get("mode", "add")))
    return out


# ---- core ops -----------------------------------------------------------------
@torch.no_grad()
def generate(prompts, system_prompt=None, max_new_tokens=200, temperature=0.0, batch=8):
    outs = []
    for i in range(0, len(prompts), batch):
        enc = encode([chat(p, system_prompt) for p in prompts[i:i + batch]])
        kw = dict(do_sample=True, temperature=temperature, top_p=0.95) if temperature > 0 else dict(do_sample=False)
        g = model.generate(**enc, max_new_tokens=max_new_tokens, pad_token_id=tok.pad_token_id, **kw)
        outs += tok.batch_decode(g[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    return outs


_mmlu, _gsm = None, None


def load_evals():
    global _mmlu, _gsm
    if _mmlu is None:
        from datasets import load_dataset
        m = list(load_dataset("cais/mmlu", "all", split="test"))
        random.Random(0).shuffle(m)
        _mmlu = m[:200]
        g = list(load_dataset("openai/gsm8k", "main", split="test"))
        random.Random(0).shuffle(g)
        _gsm = g[:100]
    return _mmlu, _gsm


LETTERS = [tok.encode(c, add_special_tokens=False)[0] for c in "ABCD"]


@torch.no_grad()
def mmlu_acc(items, system_prompt=None, batch=8):
    right = 0
    for i in range(0, len(items), batch):
        chunk = items[i:i + batch]
        texts = [chat(q["question"] + "\n" + "\n".join(f"{c}. {a}" for c, a in zip("ABCD", q["choices"]))
                      + "\nAnswer with the letter only.", system_prompt) for q in chunk]
        logits = model(**encode(texts), logits_to_keep=1).logits[:, -1, LETTERS]  # last token only; full logits OOM
        right += sum(int(p == q["answer"]) for p, q in zip(logits.argmax(-1).tolist(), chunk))
    return right / len(items)


def last_number(s):
    nums = re.findall(r"-?\d[\d,]*\.?\d*", s)
    return nums[-1].replace(",", "").rstrip(".") if nums else None


def gsm_acc(items, system_prompt=None):
    outs = generate([q["question"] + "\nThink step by step, then give the final number on the last line." for q in items],
                    system_prompt, max_new_tokens=320, batch=8)
    gold = [q["answer"].split("####")[-1].strip().replace(",", "") for q in items]
    return sum(last_number(o) == g for o, g in zip(outs, gold)) / len(items)


# ---- API ------------------------------------------------------------------------
app = FastAPI()


class Steer(BaseModel):
    layer: int
    neuron: int
    dose: float
    mode: str = "add"


class GenReq(BaseModel):
    prompts: list[str]
    steers: list[Steer] = []
    system_prompt: str | None = None
    max_new_tokens: int = 200
    temperature: float = 0.0


class CapReq(BaseModel):
    steers: list[Steer] = []
    system_prompt: str | None = None
    full: bool = False


class ContrastReq(BaseModel):
    positive: list[str]
    negative: list[str]
    top_k: int = 40
    layers: list[int] | None = None


@app.get("/health")
def health():
    return {"ok": True}


@app.get("/info")
def info():
    return {"model": MODEL_ID, "n_layers": N_LAYERS, "n_neurons": N_NEURONS, "dtype": str(DTYPE), "gpus": torch.cuda.device_count(),
            "median_resid_norm": MEDIAN}


@app.post("/generate")
def api_generate(r: GenReq):
    global STEERS
    with LOCK:
        STEERS = resolve([s.model_dump() for s in r.steers])
        try:
            return {"outputs": generate(r.prompts, r.system_prompt, r.max_new_tokens, r.temperature)}
        finally:
            STEERS = []


@app.post("/capability")
def api_capability(r: CapReq):
    """fast: 100 MMLU (logits only, ~1 min). full: 200 MMLU + 100 GSM8K (several minutes)."""
    global STEERS
    mmlu, gsm = load_evals()
    with LOCK:
        STEERS = resolve([s.model_dump() for s in r.steers])
        try:
            res = {"mmlu": mmlu_acc(mmlu if r.full else mmlu[:100], r.system_prompt)}
            if r.full:
                res["gsm8k"] = gsm_acc(gsm, r.system_prompt)
            return res
        finally:
            STEERS = []


@app.post("/contrast")
def api_contrast(r: ContrastReq):
    """Neurons whose mean down_proj input differs most between positive and negative texts
    (texts are fed raw, not chat-formatted). Score = mean difference / pooled std."""
    layers = r.layers or list(DOWN)
    acts = {L: {"pos": [], "neg": []} for L in layers}
    side = {"v": "pos"}
    def grab(L):
        def h(mod, args):
            acts[L][side["v"]].append(args[0][0, 1:].float().mean(0).cpu())
        return h
    with LOCK, torch.no_grad():
        hs = [DOWN[L].register_forward_pre_hook(grab(L)) for L in layers]
        try:
            for side["v"], texts in (("pos", r.positive), ("neg", r.negative)):
                for t in texts:
                    model(**encode([t]))
        finally:
            for h in hs:
                h.remove()
    found = []
    for L in layers:
        p, n = torch.stack(acts[L]["pos"]), torch.stack(acts[L]["neg"])
        diff = p.mean(0) - n.mean(0)
        score = diff / (torch.cat([p, n]).std(0) + 1e-6)
        for idx in score.abs().topk(r.top_k).indices.tolist():
            found.append({"layer": L, "neuron": idx, "score": round(score[idx].item(), 3),
                          "pos_mean": round(p.mean(0)[idx].item(), 4), "neg_mean": round(n.mean(0)[idx].item(), 4)})
    found.sort(key=lambda f: -abs(f["score"]))
    return {"neurons": found[:r.top_k]}

In [ ]:
%%writefile /kaggle/working/prompts.json
{
 "calibration": [
  "What is the capital of Australia?",
  "Explain how a refrigerator works.",
  "Write a haiku about rain.",
  "Give me three tips for a job interview.",
  "What causes the seasons on Earth?",
  "Summarize the plot of Romeo and Juliet.",
  "How do I make a cup of pour-over coffee?",
  "What is a prime number?",
  "Translate 'good morning' into Spanish and French.",
  "Why is the sky blue?",
  "Write a short thank-you note to a coworker.",
  "What are the pros and cons of electric cars?",
  "Explain recursion to a beginner.",
  "List five healthy breakfast ideas.",
  "How does compound interest work?",
  "What is photosynthesis?",
  "Describe the water cycle.",
  "Write a two-sentence horror story.",
  "How do vaccines work?",
  "What is the difference between a virus and bacteria?",
  "Give me a packing list for a weekend trip.",
  "Explain what an API is.",
  "What happened during the French Revolution?",
  "How do I change a flat tire?",
  "Write a limerick about a cat.",
  "What is machine learning?",
  "Explain supply and demand.",
  "How many planets are in the solar system?",
  "What should I consider when adopting a dog?",
  "Describe a sunset in one paragraph.",
  "What is the Pythagorean theorem?",
  "How do airplanes stay in the air?",
  "Write a polite email declining a meeting.",
  "What are black holes?",
  "Give me a simple pasta recipe.",
  "Explain how the stock market works.",
  "What is the boiling point of water at sea level?",
  "How do I start learning the guitar?",
  "Explain DNA in simple terms.",
  "What is the tallest mountain in the world?",
  "Write a motivational quote.",
  "How does GPS work?",
  "What are the main causes of climate change?",
  "Describe the rules of chess briefly.",
  "How do I improve my sleep?",
  "What is inflation?",
  "Explain the difference between weather and climate.",
  "Write a short poem about the ocean.",
  "How does a bill become a law in the US?",
  "What is the speed of light?",
  "Give me tips for public speaking.",
  "What is a neural network?",
  "How do bees make honey?",
  "Write a product description for a water bottle.",
  "What is the Great Wall of China?",
  "How do I budget my monthly expenses?",
  "Explain what a black box test is.",
  "What are renewable energy sources?",
  "How do I write a good resume?",
  "What is the theory of evolution?",
  "Describe how to plant a tomato.",
  "What is a sonnet?",
  "How do magnets work?",
  "What are some good study habits?"
 ],
 "heldout": [
  "I just got a promotion at work!",
  "My cat knocked my coffee off the table again.",
  "Can you help me plan a birthday party for my friend?",
  "What's the best way to learn a new language?",
  "I'm thinking about running a marathon. Thoughts?",
  "Explain how a microwave works.",
  "My code finally compiled after three hours.",
  "What should I cook for dinner tonight?",
  "Is it worth buying a standing desk?",
  "I failed my driving test today.",
  "Tell me about the Roman Empire.",
  "How do I ask my boss for a raise?",
  "What's your opinion on pineapple on pizza?",
  "I'm moving to a new city next month.",
  "Recommend a book for a long flight.",
  "Why do people like reality TV?",
  "I spilled red wine on a white carpet. Help.",
  "What's the point of learning calculus?",
  "My friend didn't reply to my text for two days.",
  "How do I keep houseplants alive?",
  "Explain blockchain like I'm five.",
  "I'm nervous about my presentation tomorrow.",
  "What's a good name for a goldfish?",
  "Should I get up early or stay up late to be productive?",
  "My laptop battery dies in an hour.",
  "What do you think about Mondays?",
  "I won a small raffle prize today.",
  "How do I deal with a noisy neighbor?",
  "Give me a fun fact about octopuses.",
  "I'm learning to bake bread. Any tips?"
 ],
 "system_prompts": {
  "sarcastic": "You are a sarcastic assistant. Answer helpfully, but with dry, sarcastic wit.",
  "chill friend": "You are a laid-back, chill friend. Talk casually and warmly, like a relaxed buddy, while still being helpful."
 },
 "lyra_steers": {
  "sarcastic": [
   {
    "layer": 15,
    "neuron": 3920,
    "dose": 1.0
   }
  ],
  "chill friend": [
   {
    "layer": 11,
    "neuron": 6261,
    "dose": 0.4
   },
   {
    "layer": 17,
    "neuron": 7870,
    "dose": -0.4
   }
  ]
 }
}

In [ ]:
%%writefile /kaggle/working/neurons.py
"""Steer strings in lyra's syntax ("L15N3920+0.6 L12N2920-0.5") and the playground presets."""
import glob, json, random, re

TOKEN = re.compile(r"L(\d+)N(\d+)([+-](?:\d+\.?\d*|\.\d+))?", re.I)

PRESETS = {
    "sarcastic": "L15N3920+1",
    "sarcastic + anti-sycophancy": "L15N3920+1 L12N2920-1",
    "condolences on good things": "L10N6295+1",
    "chill friend": "L11N6261+0.4 L17N7870-0.4",
    "lyra mix": "L18N5796+0.6 L15N3920+0.6 L12N2920-0.5 L10N6295+0.4 L16N8168-0.3",
    # From lyra's neuron-explorer research_log. Doses there are given both in abs_max units (N2742 "+2" = 0.74x residual)
    # and in residual-norm units; ours are residual-norm units, so these use the latter.
    "alien identity": "L22N2742+0.8",
    "responsible AI persona": "L22N5344-0.8",
}


def parse_steers(text, n_layers=None, n_neurons=None):
    """'L15N3920+0.6 L12N2920-0.5' -> [{"layer": 15, "neuron": 3920, "dose": 0.6}, ...].
    Tokens are split on whitespace or commas; a token with no dose means +1."""
    steers = []
    for tok in re.split(r"[\s,]+", (text or "").strip()):
        if not tok:
            continue
        m = TOKEN.fullmatch(tok)
        if not m:
            raise ValueError(f"can't read {tok!r}; write steers like L15N3920+0.6")
        L, n = int(m.group(1)), int(m.group(2))
        if n_layers is not None and L >= n_layers:
            raise ValueError(f"{tok}: layer {L} is out of range (model has {n_layers} layers)")
        if n_neurons is not None and n >= n_neurons:
            raise ValueError(f"{tok}: neuron {n} is out of range (layers have {n_neurons} neurons)")
        steers.append({"layer": L, "neuron": n, "dose": float(m.group(3)) if m.group(3) else 1.0})
    return steers


def format_steers(steers):
    return " ".join(f"L{s['layer']}N{s['neuron']}{s['dose']:+g}" for s in steers)


def scale(steers, mult):
    return [{**s, "dose": s["dose"] * mult} for s in steers]


def control(steers, kind, n_neurons, seed=0):
    """Controls for a steer: "sign flip" negates every dose; "random neuron" swaps each neuron for a random one in
    the same layer at the same dose (same push size, since doses are in residual-norm units)."""
    if kind == "sign flip":
        return scale(steers, -1)
    if kind == "random neuron":
        rng = random.Random(seed)
        return [{**s, "neuron": rng.randrange(n_neurons)} for s in steers]
    return steers


def load_presets(pattern="/kaggle/input/**/steers.json"):
    """Lyra's presets plus 'agent: <trait>' for each trait in any attached steers.json."""
    presets = {name: parse_steers(s) for name, s in PRESETS.items()}
    for path in sorted(glob.glob(pattern, recursive=True)):
        with open(path) as f:
            agent = json.load(f)
        for trait, steers in agent.items():
            if steers:
                presets[f"agent: {trait}"] = [{"layer": int(s["layer"]), "neuron": int(s["neuron"]),
                                               "dose": float(s["dose"])} for s in steers]
    return presets

In [ ]:
%%writefile /kaggle/working/card.py
"""Screenshot-ready HTML card for one question: grey Normal Qwen box, accent steered box(es), footer with the setup.
Model text is only escaped and, past MAX_CHARS, cut with "…"; the full text goes to the copy box."""
import html

MAX_CHARS = 700
GREY = "#8a8f98"
ACCENTS = ["#e8590c", "#1c7ed6", "#2f9e44", "#ae3ec9", "#f08c00", "#0c8599", "#d6336c", "#5c7cfa"]
CUSTOM_ACCENT = "#495057"

CSS = """
.sc-card { font-family: Inter, system-ui, sans-serif; background: #fff; color: #111; padding: 28px; border-radius: 16px; }
.sc-title { font-size: 40px; font-weight: 700; line-height: 1.2; margin: 0 0 20px; }
.sc-box { font-size: 24px; line-height: 1.4; border-radius: 12px; padding: 16px 20px; margin: 0 0 14px;
          border-left: 10px solid var(--c); background: color-mix(in srgb, var(--c) 10%, #fff); white-space: pre-wrap; }
.sc-box.sc-ctrl { background: repeating-linear-gradient(135deg, #f1f3f5 0 14px, #e9ecef 14px 28px); }
.sc-tag { display: block; font-size: 20px; font-weight: 700; color: var(--c); margin-bottom: 6px; }
.sc-hits { font-size: 28px; font-weight: 700; margin: 4px 0 14px; }
.sc-foot { font-size: 18px; color: #555; margin-top: 8px; }
"""


def accent_for(preset, presets):
    names = list(presets)
    return ACCENTS[names.index(preset) % len(ACCENTS)] if preset in names else CUSTOM_ACCENT


def steer_tag(steers):
    """[{"layer": 15, "neuron": 3920, "dose": 1.0}] -> "L15 N3920 +1.0"; mode "set" (clamp) shows as "L15 N3920 =1.0"."""
    return ", ".join(f"L{s['layer']} N{s['neuron']} " + (f"={s['dose']:.1f}" if s.get("mode") == "set" else f"{s['dose']:+.1f}")
                     for s in steers) or "no steers"


def clip(text, n=MAX_CHARS):
    text = (text or "").strip()
    return text if len(text) <= n else text[:n].rstrip() + "…"


def box(tag, text, color, ctrl=False):
    return (f'<div class="sc-box{" sc-ctrl" if ctrl else ""}" style="--c:{color}">'
            f'<span class="sc-tag">{html.escape(tag)}</span>{html.escape(clip(text))}</div>')


def render(question, unsteered, steered, tag, color, footer, ctrl=False, hits=None):
    """unsteered: one reply; steered: list of replies (1 or 3); hits: optional summary line like "2 of 3 sarcastic"."""
    parts = [f'<div class="sc-card"><div class="sc-title">{html.escape(clip(question, 200))}</div>',
             box("Normal Qwen", unsteered, GREY)]
    many = len(steered) > 1
    for i, s in enumerate(steered):
        parts.append(box(f"Neuron Qwen · {tag}" + (f" · sample {i + 1}" if many else ""), s, color, ctrl))
    if hits:
        parts.append(f'<div class="sc-hits">{html.escape(hits)}</div>')
    parts.append(f'<div class="sc-foot">{html.escape(footer)}</div></div>')
    return "".join(parts)


def full_text(question, unsteered, steered, tag, footer):
    """Plain text of everything, untruncated, for the copy box."""
    out = [f"Q: {question}", "", "[Normal Qwen]", unsteered.strip()]
    for i, s in enumerate(steered):
        out += ["", f"[Neuron Qwen · {tag}" + (f" · sample {i + 1}]" if len(steered) > 1 else "]"), s.strip()]
    return "\n".join(out + ["", footer])

In [ ]:
# Loads the model and calibrates the steer scale (first run downloads ~9 GB of weights).
import os, sys, threading, torch
os.chdir("/kaggle/working"); sys.path.insert(0, "/kaggle/working")
os.environ["DTYPE"] = DTYPE
import neuron_server as ss
from neurons import control, load_presets, parse_steers, format_steers, scale
import card

PRESETS = load_presets()
print("presets:", {k: format_steers(v) for k, v in PRESETS.items()})
GEN_LOCK = threading.Lock()
TEMP = 0.7
YES_NO = [ss.tok.encode(w, add_special_tokens=False)[0] for w in ("yes", "Yes", "no", "No")]


def run(prompt, steers, max_new_tokens, n=1, seed=0):
    """n=1: one greedy reply. n>1: n samples at T=0.7 from a fixed seed."""
    with GEN_LOCK:
        ss.STEERS = ss.resolve(steers)
        try:
            if n == 1:
                return [ss.generate([prompt], max_new_tokens=int(max_new_tokens))[0].strip()]
            torch.manual_seed(seed)
            return [o.strip() for o in ss.generate([prompt] * n, max_new_tokens=int(max_new_tokens), temperature=TEMP)]
        finally:
            ss.STEERS = []


@torch.no_grad()
def shows_trait(reply, trait):
    """Unsteered Qwen judges one reply: P(yes) > P(no) for 'does this come across as <trait>?'."""
    q = f"Here is a chatbot reply:\n\n{reply[:2000]}\n\nDoes this reply come across as {trait}? Answer yes or no."
    with GEN_LOCK:
        logits = ss.model(**ss.encode([ss.chat(q)]), logits_to_keep=1).logits[0, -1, YES_NO].float()
    return bool(logits[:2].logsumexp(0) > logits[2:].logsumexp(0))


def answer(question, preset, mult, custom, max_new_tokens, ctrl="none", seed=0, samples=1, trait="", mode="add"):
    """Returns (card html, full text for copying)."""
    if not (question or "").strip():
        return "<p>Type a question first.</p>", ""
    is_custom = bool((custom or "").strip())
    try:
        base = parse_steers(custom, ss.N_LAYERS, ss.N_NEURONS) if is_custom else PRESETS[preset]
    except ValueError as e:
        return f"<p><b>Steer error:</b> {e}</p>", ""
    seed, n = int(seed), int(samples)
    steers = [{**s, "mode": mode} for s in control(scale(base, mult), ctrl, ss.N_NEURONS, seed)]
    name = "custom" if is_custom else preset
    tag = card.steer_tag(steers) + ("" if ctrl == "none" else f" (control: {ctrl})")
    unsteered = run(question, [], max_new_tokens, n, seed)
    steered = run(question, steers, max_new_tokens, n, seed)
    hits = None
    ctrl_note = "" if ctrl == "none" else f"control: {ctrl}" + (f", seed {seed}" if ctrl == "random neuron" else "") + " · "
    if n == 1:
        decoding = "greedy · single sample"
    else:
        trait = (trait or "").strip() or name
        k_s = sum(shows_trait(r, trait) for r in steered)
        k_u = sum(shows_trait(r, trait) for r in unsteered)
        hits = f"Neuron Qwen: {k_s} of {n} {trait} · Normal Qwen: {k_u} of {n}"
        decoding = f"T={TEMP} · {n} samples, seed {seed} · judged by unsteered Qwen3.5-4B"
    footer = f"Qwen3.5-4B · {name} x{mult:g} · {card.steer_tag(steers)} ({mode}) · {ctrl_note}{decoding}"
    color = card.CUSTOM_ACCENT if is_custom else card.accent_for(preset, PRESETS)
    return (card.render(question, unsteered[0], steered, tag, color, footer, ctrl != "none", hits),
            card.full_text(question, unsteered[0], steered, tag, footer))


print(run("Say hi in five words.", [], 20))
print(run("Say hi in five words.", PRESETS["sarcastic"], 20))

In [ ]:
# Fallback if the gradio.live link is blocked: the same controls right here in the notebook (interactive sessions only).
# Or call it directly: ask("Why is the sky blue?", preset="sarcastic", mult=1.5, samples=3)
from IPython.display import display, HTML
display(HTML(f"<style>{card.CSS}</style>"))

def ask(question, preset="sarcastic", mult=1.0, custom="", max_new_tokens=200, ctrl="none", seed=0, samples=1, trait="", mode="add"):
    html_card, full = answer(question, preset, mult, custom, max_new_tokens, ctrl, seed, samples, trait, mode)
    display(HTML(html_card))
    return full

try:
    import ipywidgets as w
    q = w.Textarea(placeholder="Ask it something silly", layout=w.Layout(width="100%"))
    p = w.Dropdown(options=list(PRESETS), value="sarcastic", description="Preset")
    m = w.FloatSlider(value=1, min=0, max=3, step=0.1, description="Dose x")
    c = w.Text(placeholder="L15N3920+0.6 L12N2920-0.5", description="Custom")
    t = w.IntSlider(value=200, min=32, max=512, step=16, description="Max tokens")
    k = w.Dropdown(options=["none", "random neuron", "sign flip"], value="none", description="Control")
    s = w.Dropdown(options=[("1 (greedy)", 1), ("3 samples (T=0.7)", 3)], value=1, description="Samples")
    md = w.Dropdown(options=["add", "set"], value="add", description="Mode")
    b, out = w.Button(description="Ask", button_style="primary"), w.Output()
    def on_ask(_):
        out.clear_output()
        with out:
            print(ask(q.value, p.value, m.value, c.value, t.value, k.value, 0, s.value, "", md.value))
    b.on_click(on_ask)
    display(w.VBox([q, w.HBox([p, m]), w.HBox([c, t]), w.HBox([k, s, md]), b, out]))
except Exception as e:
    print("widgets unavailable, use ask(...) instead:", e)

In [ ]:
import gradio as gr

with gr.Blocks(title="Neuron playground") as demo:
    gr.Markdown("## Qwen3.5 4B: Normal Qwen vs Neuron Qwen")
    question = gr.Textbox(label="Question", placeholder="Ask it something silly", lines=2)
    with gr.Row():
        preset = gr.Dropdown(list(PRESETS), value="sarcastic", label="Preset")
        mult = gr.Slider(0, 3, value=1, step=0.1, label="Dose multiplier",
                         info="Replies usually turn to gibberish once a push passes ~1.5x the residual norm")
        max_tok = gr.Slider(32, 512, value=200, step=16, label="Max tokens")
    custom = gr.Textbox(label="Custom neurons (override the preset)", placeholder="L15N3920+0.6 L12N2920-0.5")
    with gr.Row():
        ctrl = gr.Radio(["none", "random neuron", "sign flip"], value="none", label="Control",
                        info="random neuron: same layers and push size, random neurons. sign flip: every dose negated")
        seed = gr.Number(0, precision=0, label="Seed")
    with gr.Row():
        samples = gr.Radio([("1 (greedy)", 1), ("3 samples (T=0.7)", 3)], value=1, label="Samples")
        trait = gr.Textbox(label="Trait to count (3 samples)", placeholder="defaults to the preset name")
        mode = gr.Radio(["add", "set"], value="add", label="Steer mode",
                        info="add: activation += value. set: activation clamped to the value at every token")
    go = gr.Button("Ask", variant="primary")
    out = gr.HTML()
    with gr.Accordion("Copy full text", open=False):
        full = gr.Textbox(show_label=False, lines=10, buttons=["copy"])
    inputs = [question, preset, mult, custom, max_tok, ctrl, seed, samples, trait, mode]
    go.click(answer, inputs, [out, full])
    question.submit(answer, inputs, [out, full])

try:
    _, local_url, share_url = demo.queue().launch(share=True, prevent_thread_lock=True, css=card.CSS)
    print("PUBLIC LINK:", share_url, flush=True)
except Exception as e:
    print("Gradio share link failed, use the controls in the cell above instead:", e)

In [ ]:
# Batch runs (Save & Run All) end after the last cell, which would kill the app, so hold it open for a while.
import time
if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") == "Batch":
    time.sleep(KEEP_ALIVE_HOURS * 3600)
    demo.close()